This notebook is a Exploratory Data Analysis of what Dataset 1 contains and how it can be exploited to construct a novel dataset for our task

In [1]:
import os
os.chdir("..")

In [2]:
import pandas as pd

Loading the dataset

In [3]:
dataset = pd.read_csv("./data/raw/pruning-datasets/dataset1/dataset1_gold_decisions_filtered.csv")
dataset

,QID,label,from,starting label,target,depth
0,Q26763979,mobile phone network standard,Q204833,Enhanced Data Rates for GSM Evolution,1,1
1,Q1023122,CDMA2000,Q204833,Enhanced Data Rates for GSM Evolution,1,2
2,Q920890,mobile operating system,Q483318,Symbian,1,1
3,Q4885200,Windows Phone,Q483318,Symbian,1,2
4,Q1854343,Windows Phone 7.x,Q483318,Symbian,1,3
...,...,...,...,...,...,...
4524,Q7974565,waterway society,Q37033,World Wide Web Consortium,0,2
4525,Q659206,Technischer Überwachungsverein,Q37033,World Wide Web Consortium,0,2
4526,Q20651218,Amicale du quartier,Q37033,World Wide Web Consortium,0,2
4527,Q294801,eco-control body,Q37033,World Wide Web Consortium,0,2


The dataset, such formatted, means that the target `QID`, having its correspondent `label` is (not) connected to `from` QID, which has its correspondent label `starting_label` at `depth` if the `target` is `1` (`0`).

This means that all `QID`s are Wikidata **CLASSES**.


In [4]:
len(dataset["from"].unique()) # seed qids

439

In [5]:
len(dataset[dataset["target"] == 1]["QID"].unique())

961

Let's see if there's something for which we can cool this number of classes off

In [6]:
kept = dataset[dataset["target"] == 1]
kept

,QID,label,from,starting label,target,depth
0,Q26763979,mobile phone network standard,Q204833,Enhanced Data Rates for GSM Evolution,1,1
1,Q1023122,CDMA2000,Q204833,Enhanced Data Rates for GSM Evolution,1,2
2,Q920890,mobile operating system,Q483318,Symbian,1,1
3,Q4885200,Windows Phone,Q483318,Symbian,1,2
4,Q1854343,Windows Phone 7.x,Q483318,Symbian,1,3
...,...,...,...,...,...,...
4516,Q1408288,manufacturing process,Q902751,pressurization,1,2
4517,Q20074315,boiling,Q902751,pressurization,1,2
4518,Q2992249,condition,Q902751,pressurization,1,2
4521,Q1328899,standards organization,Q37033,World Wide Web Consortium,1,1


In [7]:
rejected = dataset[dataset["target"] == 0]
rejected

,QID,label,from,starting label,target,depth
10,Q203087,signaling protocol,Q1505049,S/MIME,0,2
12,Q2622793,JetDirect,Q1505049,S/MIME,0,2
13,Q3359815,PPPoX,Q1505049,S/MIME,0,2
14,Q5013874,CRFS (Coherent Remote File System),Q1505049,S/MIME,0,2
17,Q17074885,Physiological Signal Based Security,Q1505049,S/MIME,0,2
...,...,...,...,...,...,...
4524,Q7974565,waterway society,Q37033,World Wide Web Consortium,0,2
4525,Q659206,Technischer Überwachungsverein,Q37033,World Wide Web Consortium,0,2
4526,Q20651218,Amicale du quartier,Q37033,World Wide Web Consortium,0,2
4527,Q294801,eco-control body,Q37033,World Wide Web Consortium,0,2


In [8]:
kept[["from", "depth", "target"]].groupby(by=["from", "depth"]).count()

target
from      depth        
Q1017587  1           2
Q1026962  1           2
Q10290274 1           1
          2           6
Q10322548 1           1
...                 ...
Q959710   3           6
          4           3
Q978801   1           1
          2           2
          3           1

[754 rows x 1 columns]

Check for different seeds that are instances of the same class

In [12]:
dataset[["QID", "starting label", "target"]].groupby(["starting label", "QID"]).count()["target"].unique()

array([1])

This means that each seed QID is the unique representative of a "class"

In [10]:
kept[kept["from"] == "Q483318"]

,QID,label,from,starting label,target,depth
2,Q920890,mobile operating system,Q483318,Symbian,1,1
3,Q4885200,Windows Phone,Q483318,Symbian,1,2
4,Q1854343,Windows Phone 7.x,Q483318,Symbian,1,3
5,Q34825,Windows Phone 8,Q483318,Symbian,1,3
6,Q18845397,Windows 10 Mobile,Q483318,Symbian,1,3
7,Q15282750,Windows Phone 8.1,Q483318,Symbian,1,3


Hint: There's a hierichical structure among classes. 

In [11]:
rejected[rejected["from"] == "Q483318"]

,QID,label,from,starting label,target,depth


1. **TARGET NODE TYPES**: intersaction of descriptions about kept classes - descriptions about rejected classes
2. **INSTANCES**: instances that are instance of one of the kept classes, and are not instances of rejected classes 